# Export the v2 student as one offline model file (GGUF, 4-bit)

Loads `lora_adapter_Qwen3-4B-Instruct-2507_v2` from `MyDrive/chameleon-distill/`, merges it into the base, converts it to a 4-bit GGUF on Colab's local disk, and copies only that file (about 2.5 GB) to Drive as `chameleon_student_v2.Q4_K_M.gguf`.

**Runtime → Change runtime type → L4** (or T4), then **Run all** (about 15–20 min). Make sure Drive has about 3 GB free.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil
D = '/content/drive/MyDrive/chameleon-distill'
ADAPTER = f'{D}/lora_adapter_Qwen3-4B-Instruct-2507_v2'
assert os.path.isdir(ADAPTER), f'missing {ADAPTER}'
!pip -q install unsloth

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(ADAPTER, max_seq_length=4096, load_in_4bit=False)
LOCAL = '/content/gguf_v2'
model.save_pretrained_gguf(LOCAL, tokenizer, quantization_method='q4_k_m')
found = [f for f in glob.glob(f'{LOCAL}/**/*.gguf', recursive=True) + glob.glob('/content/*.gguf') if 'q4_k_m' in f.lower()]
print(found)
dst = f'{D}/chameleon_student_v2.Q4_K_M.gguf'
shutil.copy(found[0], dst)
print('copied ->', dst, round(os.path.getsize(dst) / 1e9, 2), 'GB')